# Graphs and Graphing Algorithms


<table align="left">
  <td>
    <a href="https://colab.research.google.com/github/phonchi/nsysu-math208/blob/master/static_files/presentations/08_Graphs and Graphing Algorithms.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>
  </td>
  <td>
    <a target="_blank" href="https://kaggle.com/kernels/welcome?src=https://github.com/phonchi/nsysu-math208/blob/master/static_files/presentations/08_Graphs and Graphing Algorithms.ipynb"><img src="https://kaggle.com/static/images/open-in-kaggle.svg" /></a>
  </td>
</table>

In [ ]:
from jupyterquiz import display_quiz

quiz_path = "questions/ch8/"

In [ ]:
# === DS08 animation loader. ===
# Each animation is rendered inside a self-contained iframe.  This avoids
# relying on the notebook frontend to execute <script> tags embedded directly
# in ordinary HTML output, which is blocked or unreliable in several frontends.
from functools import lru_cache
from html import escape
from pathlib import Path
from urllib.request import urlopen
from IPython.display import display

_BASE = "https://raw.githubusercontent.com/phonchi/nsysu-math208/refs/heads/main/extra/animations08/"
_LOCAL_ANIM_DIRS = [
    Path("animations08"),
    Path("nsysu-math208/extra/animations08"),
    Path("../../extra/animations08"),
]

@lru_cache(maxsize=32)
def _fetch(name):
    for folder in _LOCAL_ANIM_DIRS:
        path = folder / name
        if path.exists():
            return path.read_text(encoding="utf-8")
    return urlopen(_BASE + name, timeout=15).read().decode("utf-8")

class _RawHTML:
    def __init__(self, html):
        self.html = html

    def _repr_html_(self):
        return self.html

def _anim_iframe(name, height=500):
    css = _fetch("ds08.css")
    js = _fetch("ds08.js").replace("</script>", "<\\/script>")
    panel = _fetch(name + ".html")
    srcdoc = f"""<!doctype html>
<html>
<head>
<meta charset=\"utf-8\">
<style>html,body{{margin:0;padding:0;background:transparent;}}</style>
<style>{css}</style>
</head>
<body class="rise-enabled">
{panel}
<script>{js}</script>
</body>
</html>"""
    return _RawHTML(
        f'<iframe title="DS08 {escape(name)} animation" '
        f'srcdoc="{escape(srcdoc, quote=True)}" '
        f'style="width:100%;height:{height}px;border:0;display:block;" '
        f'loading="lazy"></iframe>'
    )

def load_infra():
    # Kept for old cells.  The iframe loader now carries its own CSS/JS.
    return None

def load_anim(name):
    display(_anim_iframe(name))

_ = load_infra()


1. The Graph Abstract Data Type

2. Breadth-First Search

3. Depth-First Search

4. Shortest Path Problems

5. Minimum Spanning Tree


## 9.1 Introduction

<u>Graphs</u> can be used to represent many interesting things about our world, including systems of roads, airline flights, how the internet is connected, or even the sequence of classes you must take.

We will see in this chapter that once we have a good representation for a problem, we can use some standard graph algorithms to solve what otherwise might seem to be a very difficult problem!

A graph is just like a road map. If you have ever used one of the internet map sites, you know that a computer can find the shortest, quickest, or easiest path from one place to another.

## 9.2. Vocabulary and Definitions

- <u>Vertex</u>: A vertex (also called a **node**) is a fundamental part of a graph. It can have a name, which we will call the **key**. A vertex may also have additional information. We will call this additional information the **value** or the payload.

- <u>Edge</u>: An edge (also called an **arc**) is another fundamental part of a graph. An edge connects two vertices to show that there is a relationship between them. Edges may be one-way or two-way. If the edges in a graph are all one-way, we say that the graph is a <u>directed graph</u>, or a digraph.

- <u>Weight</u>: Edges may be weighted to show that there is a cost to go from one vertex to another, which we call **edge cost**. For example, in a graph of roads that connect one city to another, the weight on the edge might represent the distance between the two cities.

A graph can be represented by $G =(V,E)$. For the graph $G$, $V$ is a set of vertices and $E$ is a set of edges. Each edge is a tuple $(v,w)$ where $v, w \in V$. We can add a third component to the edge tuple to represent a weight. A subgraph $s$ 
 is a set of edges $e$ and vertices $v$ such that $e \subset E$ and $v \subset V$.

<center><img src="https://raw.githubusercontent.com/phonchi/nsysu-math208/refs/heads/main/static_files/presentations/imgs/digraph.png" width="33%" /></center>

The above is a simple weighted digraph. Formally we can represent this graph as the set of six vertices:

$$V = \left\{ v_0, v_1, v_2, v_3, v_4, v_5 \right\}$$

and the set of nine edges:

$$\begin{split}E = \left\{ \begin{array}{l}(v_0, v_1, 5), (v_1, v_2, 4), (v_2, v_3, 9), \\
                             (v_3, v_4, 7), (v_4, v_0, 1), (v_0, v_5, 2), \\
                             (v_5, v_4, 8), (v_3, v_5, 3), (v_5, v_2, 1)
             \end{array} \right\}\end{split}$$

The example graph helps illustrate two other key graph terms:

- <u>Path</u>: A path in a graph is a sequence of distinct vertices $w_1, w_2, \ldots, w_n$ such that $(w_i, w_{i+1}) \in E$ for all $1 \le i \le n-1$. The **unweighted path length** is the number of edges, $n-1$; the **weighted path length** is the sum of the weights of all edges in the path. For example, the path from $v_3$ to $v_1$ is $(v_3, v_4, v_0, v_1)$. (A more general sequence that allows repeated vertices is called a **walk**.)

- <u>Cycle</u>: A cycle is a closed sequence $(w_1, w_2, \ldots, w_n, w_1)$ where $w_1, \ldots, w_n$ are distinct vertices and consecutive vertices are connected by an edge. For example, $(v_5, v_2, v_3, v_5)$ is a cycle. A graph with no cycles is called an **acyclic graph**, and a directed graph with no cycles is called a **directed acyclic graph (DAG)**. We will see that many important problems can be solved efficiently if they can be modeled as a DAG!

A <u>tree</u> is a connected, acyclic, undirected graph. In other words, there is a path between every pair of vertices, and the graph contains no cycles. We will explore different types of trees in detail in the next chapter!

## 9.3. The Graph Abstract Data Type

Note that vertices may be either connected to each other or isolated. Edges join two vertices and may be weighted.

- `Graph()`: creates a new empty graph.

- `set_vertex(vert)`: adds an instance of Vertex to the graph.

- `add_edge(from_vert, to_vert)`: adds a new directed edge to the graph that connects two vertices.

- `add_edge(from_vert, to_vert, weight)`: adds a new weighted directed edge to the graph that connects two vertices.

- `get_vertex(vert_key)`: finds the vertex in the graph named `vert_key`.

- `get_vertices()`: returns the list of all vertices in the graph.

- `in`: returns `True` for a statement `vertex in graph` if the given vertex is in the graph, `False` otherwise.

Now that we have looked at the definition for the graph ADT, there are several ways we can implement it. There are two well-known implementations of a graph, the <u>adjacency matrix</u> and the <u>adjacency list</u>. 

## 9.4. An Adjacency Matrix

One of the easiest ways to implement a graph is to use a two-dimensional matrix. Each of the rows and columns represents a vertex in the graph. The value that is stored in the cell at the intersection of row $v$ and column $w$ indicates if there is an edge.

<center><img src="https://raw.githubusercontent.com/phonchi/nsysu-math208/refs/heads/main/static_files/presentations/imgs/adjMat.png" width="30%" /></center>

When two vertices are connected by an edge, we say that they are <u>adjacent</u> and the value in each cell represents the weight. The advantage of the adjacency matrix is that it is simple, and for small graphs it is easy to see which nodes are connected to other nodes. However, we can say that this matrix is **sparse**. A matrix is not a very efficient way to store sparse data. 

The adjacency matrix is thus a good implementation for a graph when the number of edges is large. Since there is one row and one column for every vertex in the graph, the number of edges required to fill the matrix is $|V|^2$.

However, there are few real problems that approach this sort of connectivity. The problems we will look at in this chapter all involve graphs that are sparsely connected!

## 9.5. An Adjacency List

A more space-efficient way to implement a sparsely connected graph is to use an adjacency list. In this implementation, we keep a master list of all the vertices in the `Graph` object, and each vertex object in the graph maintains a list of the other vertices that it is connected to. 

In our implementation of the `Vertex` class we will use a dictionary rather than a list, where the dictionary keys are the vertices and the values are the weights.

The advantage of the adjacency list implementation is that it allows us to compactly represent a sparse graph. The adjacency list also allows us to easily find all the links that are directly connected to a particular vertex!

<center><img src="https://raw.githubusercontent.com/phonchi/nsysu-math208/refs/heads/main/static_files/presentations/imgs/adjlist.png" width="40%" /></center>

In [ ]:
display_quiz(quiz_path+"graph.json", max_width=800)

<IPython.core.display.Javascript object>

## 9.6. Implementation

```cpp
class Vertex {
    public:
        string key;
        map<string, int> neighbors;   // neighbor key -> edge weight

        Vertex() {}
        Vertex(string k) {
            key = k;
        }

        int getNeighbor(string other) {
            if (neighbors.count(other)) {
                return neighbors[other];
            }
            return -1;   // no such edge
        }
        void setNeighbor(string other, int weight = 0) {
            neighbors[other] = weight;
        }
};
```

Each `Vertex` uses a dictionary to keep track of the vertices to which it is connected and the weight of each edge. Note that the constructor simply initializes the key, which will typically be a string and the `get_neighbor()` method returns the weight of the edge from this vertex to the vertex passed as a parameter.

```cpp
class Graph {
    public:
        map<string, Vertex> vertices;

        void setVertex(string key) {
            vertices[key] = Vertex(key);
        }
        void addEdge(string fromVert, string toVert, int weight = 0) {
            if (vertices.count(fromVert) == 0) setVertex(fromVert);
            if (vertices.count(toVert) == 0) setVertex(toVert);
            vertices[fromVert].neighbors[toVert] = weight;
        }
        bool contains(string key) {
            return vertices.count(key) > 0;
        }
};
```

The `Graph` class, also contains a dictionary that **maps vertex names to vertex objects**. Note that we have implemented the `__iter__` method to make it easy to iterate over all the vertex objects in a particular graph. 

Now let us define the graph that we have seen earlier. First we create six vertices numbered 0 through 5. Then we display the vertex dictionary. Notice that for each key 0 through 5 we have created an instance of a `Vertex`:

In [1]:
#include <iostream>
#include "dscpp/graph.hpp"   // Vertex + Graph (md listing above)
using namespace std;

int main() {
    Graph g;
    for (int i = 0; i < 6; i++) {
        g.setVertex(to_string(i));
    }
    for (auto& p : g.vertices) {
        cout << "Vertex(" << p.first << ") ";
    }
    cout << endl;
    return 0;
}

Vertex(0) Vertex(1) Vertex(2) Vertex(3) Vertex(4) Vertex(5) 



Next, we add the edges that connect the vertices together. Finally, a nested loop verifies that each edge in the graph is properly stored:

In [2]:
#include <iostream>
#include "dscpp/graph.hpp"
using namespace std;

int main() {
    Graph g;
    g.addEdge("0","1",5); g.addEdge("0","5",2); g.addEdge("1","2",4);
    g.addEdge("2","3",9); g.addEdge("3","4",7); g.addEdge("3","5",3);
    g.addEdge("4","0",1); g.addEdge("5","4",8); g.addEdge("5","2",1);

    for (auto& p : g.vertices)
        for (auto& n : p.second.neighbors)
            cout << "(" << p.first << "," << n.first << "," << n.second << ") ";
    cout << endl;
    return 0;
}

(0,1,5) (0,5,2) (1,2,4) (2,3,9) (3,4,7) (3,5,3) (4,0,1) (5,2,1) (5,4,8) 



https://visualgo.net/en/graphds

## 9.7. The Word Ladder Problem

To begin our study of graph algorithms let's consider the following puzzle called a **word ladder**: transform the word FOOL into the word SAGE. In a word ladder puzzle you must make the change occur gradually by changing **one letter** at a time. 

At each step you must transform one word into another word; **you are not allowed to transform a word into a non-word**. The following sequence of words shows one possible solution to the problem posed above:

```
FOOL
POOL
POLL
POLE
PALE
SALE
SAGE
```

There are many variations of the word ladder puzzle. For example you might be given a particular number of steps in which to accomplish the transformation, or you might need to use a particular word. In this section, we are interested in figuring out the smallest number of transformations needed to turn the starting word into the ending word.

Here is an outline of where we are going:

- Represent the relationships between the words as a graph.

- Use the graph algorithm known as <u>breadth-first search</u> to find an efficient path from the starting word to the ending word.

## 9.8. Building the Word Ladder Graph

Our first problem is to figure out how to turn a large collection of words into a graph. What we would like is to have an edge from one word to another if the two words are only different by a single letter.

If we can create such a graph, then any path from one word to another is a solution to the word ladder puzzle!

<center><img src="https://raw.githubusercontent.com/phonchi/nsysu-math208/refs/heads/main/static_files/presentations/imgs/wordgraph.png" width="55%" /></center>

Let's start with the assumption that we have a list of words that are all the same length. As a starting point, we can create a vertex in the graph for every word in the list.

 To figure out how to connect the words, we could compare each word in the list with every other. When we compare we are looking to see how many letters are different. If the two words in question are different by only one letter, we can create an edge between them in the graph. 

For a small set of words that approach would work fine. However, let's suppose we have a list of [5,110 words](https://wordsrated.com/tools/wordslists/4-letter-words/). Roughly speaking, comparing one word to every other word on the list is an $O(n^2)$ algorithm. For 5,110 words, is more than 26 million comparisons!

We can do much better by assuming that we have a number of buckets, each labeled with a four-letter word, except that one of the letters on the label has been replaced by an underscore. As we process a list of words, we compare each word with each bucket using the underscore `(_)` as a wildcard.

Every time we find a matching bucket we put the word in that bucket, so that both `POPE` and `POPS` would both go into the `POP_` bucket. Once we have all the words in the appropriate buckets, we know that all the words in each bucket must be connected!

<center><img src="https://raw.githubusercontent.com/phonchi/nsysu-math208/refs/heads/main/static_files/presentations/imgs/wordbuckets.png" width="35%" /></center>

The graph classes are collected in `pythonds3/cppds/graphs/adjacency_graph.cpp`.

```cpp
Graph buildGraph(vector<string> words) {
    map<string, set<string>> buckets;
    Graph theGraph;

    // create buckets of words that differ by one letter
    for (string word : words) {
        for (unsigned i = 0; i < word.length(); i++) {
            string bucket = word.substr(0, i) + "_" + word.substr(i + 1);
            buckets[bucket].insert(word);
        }
    }

    // add edges between different words in the same bucket
    for (auto& pair : buckets) {
        for (const string& word1 : pair.second) {
            for (const string& word2 : pair.second) {
                if (word1 != word2) {
                    theGraph.addEdge(word1, word2);
                }
            }
        }
    }
    return theGraph;
}
```

We can implement the scheme we have just described by using a dictionary. The labels on the buckets we have just described are the keys in our dictionary. The value stored for each key is a list of words.

Since this is our first real-world graph problem, you might be wondering how sparse the graph is. The list of four-letter words we have for this problem is 5,110 words long. If we were to use an adjacency matrix, the matrix would have 
 $5,110 \cdot 5,110= 26,112,100$ cells. The graph constructed by the `build_graph()` function has exactly 53,286 edges, so the matrix would have only 0.20% of the cells filled! That is a very sparse matrix indeed.

## 9.9. Implementing Breadth-First Search

With the graph constructed we can now turn our attention to the algorithm we will use to find the shortest solution to the word ladder problem. The graph algorithm we are going to use is called the <u>breadth-first search (BFS)</u>, and it is one of the easiest algorithms for searching a graph!

Given a starting vertex $s$ of a graph, a breadth first search proceeds by exploring edges in the graph to find all the vertices in $G$ for which there is a path from $s$. 

The remarkable thing about a breadth-first search is that it finds all the vertices that are a distance $k$ from $s$ before it finds any vertices that are a distance $k+1$. 

To keep track of its progress, BFS colors each of the vertices white, gray, or black. All the vertices are initialized to white when they are constructed. A white vertex is an undiscovered vertex. When a vertex is initially discovered it is colored gray, and when BFS has completely explored a vertex it is colored black.

This means that once a vertex is colored black, it has no white vertices adjacent to it. A gray node, on the other hand, may have some white vertices adjacent to it, indicating that there are still additional vertices to explore.

The `bfs()` shown below uses the adjacency list graph representation we developed earlier. In addition it uses a `Queue`, a crucial point as we will see, to decide which vertex to explore next.

```cpp
void bfs(Graph& g, string startKey) {
    g.vertices[startKey].distance = 0;
    g.vertices[startKey].previous = "";
    queue<string> vertQueue;
    vertQueue.push(startKey);
    while (!vertQueue.empty()) {
        string currentKey = vertQueue.front();
        vertQueue.pop();
        Vertex& current = g.vertices[currentKey];
        for (auto& n : current.neighbors) {
            Vertex& neighbor = g.vertices[n.first];
            if (neighbor.color == "white") {
                neighbor.color = "gray";
                neighbor.distance = current.distance + 1;
                neighbor.previous = currentKey;
                vertQueue.push(n.first);
            }
        }
        current.color = "black";
    }
}
```

The BFS algorithm uses an extended version of the `Vertex` class that adds three new instance variables: `distance`, `previous`, and `color`. Each of these instance variables also has the appropriate getter and setter methods. 

BFS begins at the starting vertex `start` and paints it gray. Two other values, the `distance` and the `previous`, are initialized to 0 and `None` respectively. Finally, `start` is placed on a `Queue`. The next step is to begin to systematically explore vertices at the front of the queue. We explore each new node at the front of the queue by iterating over its adjacency list.

As each node on the adjacency list is examined, its color is checked. If it is white four things happen:

1. The new unexplored vertex `neighbor` is colored gray.

2. The predecessor of `neighbor` is set to the current node `current`.

3. The distance to `neighbor` is set to the distance to `current + 1`.

4. `neighbor` is added to the end of a queue. This effectively schedules this node for further exploration, but not until all the other vertices on the adjacency list of current have been explored!

<center><img src="https://raw.githubusercontent.com/phonchi/nsysu-math208/refs/heads/main/static_files/presentations/imgs/bfs1.png" width="35%" /></center>

Starting from `FOOL` we take all nodes that are adjacent to `FOOL` and add them to the queue. The adjacent nodes include `POOL`, `FOIL`, `FOUL`, and `COOL`. Each of these nodes are new nodes to expand.

<center><img src="https://raw.githubusercontent.com/phonchi/nsysu-math208/refs/heads/main/static_files/presentations/imgs/bfs2.png" width="33%" /></center>

In the next step bfs removes the next node (`POOL`) from the front of the queue and repeats the process for all of its adjacent nodes.

However, when bfs examines the node `COOL`, it finds that the color of `COOL` has already been changed to gray. This indicates that there is a shorter path to `COOL` and that `COOL`is already on the queue for further expansion. The only new node added to the queue while examining `POOL` is `POLL`. 

The next vertex on the queue is `FOIL`. The only new node that `FOIL` can add to the tree is `FAIL`. As bfs continues to process the queue, neither of the next two nodes adds anything new to the queue or the tree. Below shows the tree and the queue after expanding all the vertices on the second level of the tree:

<center><img src="https://raw.githubusercontent.com/phonchi/nsysu-math208/refs/heads/main/static_files/presentations/imgs/bfs3.png" width="33%" /></center>

You should continue to work through the algorithm on your own so that you are comfortable with how it works. Figure below shows the final breadth-first search tree after all the vertices.

<center><img src="https://raw.githubusercontent.com/phonchi/nsysu-math208/refs/heads/main/static_files/presentations/imgs/bfsDone.png" width="33%" /></center>

The amazing thing about the breadth-first search solution is that we have not only solved the `FOOL–SAGE` problem we started out with, but we have solved many other problems along the way.

We can start at any vertex in the breadth-first search tree and follow the predecessor arrows back to the root to find the shortest word ladder from any word back to `FOOL`. 

```cpp
void traverse(Graph& g, string startingKey) {
    string current = startingKey;
    while (current != "") {
        cout << current;
        if (g.vertices[current].previous != "") {
            cout << "->";
        }
        current = g.vertices[current].previous;
    }
    cout << endl;
}
```

In [3]:
#include <iostream>
#include "dscpp/graph_algos.hpp"   // buildGraph + bfs + traverse
using namespace std;

int main() {
    Graph g = buildGraph({"fool", "cool", "pool", "poll", "pole",
                         "pall", "fall", "fail", "foil", "foul",
                         "pope", "pale", "sale", "sage", "page"});
    g.vertices["fool"].color = "gray";
    bfs(g, "fool");
    traverse(g, "sage");
    return 0;
}

sage->page->pale->pall->poll->pool->fool



In [4]:
#include <iostream>
#include "dscpp/graph_algos.hpp"
using namespace std;

int main() {
    Graph g = buildGraph({"fool", "cool", "pool", "poll", "pole",
                         "pall", "fall", "fail", "foil", "foul",
                         "pope", "pale", "sale", "sage", "page"});
    bfs(g, "fool");
    for (auto& p : g.vertices)   // word(distance from fool)
        cout << p.first << "(" << p.second.distance << ") ";
    cout << endl;
    return 0;
}

cool(1) fail(2) fall(3) foil(1) fool(0) foul(1) page(5) pale(4) pall(3) pole(3) poll(2) pool(1) pope(4) sage(6) sale(5) 



In [ ]:
load_anim("bfs")

## 9.10. Breadth-First Search Analysis

Let's analyze the run time performance of the breadth-first search algorithm. The first thing to observe is that the `while` loop is executed, at most, one time for each vertex in the graph (up to $|V|$ iterations). You can see that this is true because a vertex must be white before it can be examined and added to the queue.

This gives us $O(|V|)$ for the while loop. The `for` loop, which is nested inside the `while`, is executed at most once for each edge in the graph (up to $|E|$ iterations). The reason is that every vertex is dequeued at most once and we examine an edge from node $u$ to node $v$ only when node $u$ is dequeued.

This gives us $O(|E|)$ for the `for` loop. Combining the two loops gives us $O(|V| + |E|)$.

Of course doing the breadth-first search is only part of the task. Following the links from the starting node to the goal node is the other part of the task.

The worst case for this would be if the graph was a single long chain. In this case traversing through all of the vertices would be $O(|V|)$. The normal case is going to be some fraction of $|V|$.

Finally, at least for this problem, there is the time required to build the initial graph.

## 9.11. The Knight's Tour Problem

Another classic problem that we can use to illustrate a second common graph algorithm is called the <U>knight's tour</U>. The knight's tour puzzle is played on a chess board with a single chess piece, the knight. 

The object of the puzzle is to find a sequence of moves that allow the knight to visit every square on the board exactly once. One such sequence is called a **tour**. 

The upper bound on the number of possible legal tours for an $8 \times 8$ chessboard is known to be [$10^{35}$](https://en.wikipedia.org/wiki/Knight%27s_tour); however, there are even more possible dead ends.

Although researchers have studied many different algorithms to solve the knight's tour problem, a graph search is one of the easiest to understand and program. Once again we will solve the problem using two main steps:

- Represent the legal moves of a knight on a chessboard as a graph.

- Use a graph algorithm to find a path of length $rows \times columns - 1$ where every vertex on the graph is visited exactly once!

## 9.12. Building the Knight's Tour Graph

To represent the knight's tour problem as a graph we will use the following two ideas: each square on the chessboard can be represented as a node in the graph and each legal move by the knight can be represented as an edge in the graph.

<center><img src="https://raw.githubusercontent.com/phonchi/nsysu-math208/refs/heads/main/static_files/presentations/imgs/knightmoves.png" width="45%" /></center>

```cpp
Graph knightGraph(int boardSize) {
    Graph ktGraph;
    for (int row = 0; row < boardSize; row++) {
        for (int col = 0; col < boardSize; col++) {
            int nodeId = row * boardSize + col;
            for (auto& move : genLegalMoves(row, col, boardSize)) {
                int otherNodeId = move.first * boardSize + move.second;
                ktGraph.addEdge(to_string(nodeId), to_string(otherNodeId));
            }
        }
    }
    return ktGraph;
}
```

The `knight_grah()` function makes one pass over the entire board. At each square on the board the function calls a helper, `gen_legal_moves()`, to create a list of legal moves for that position on the board.

All legal moves are then converted into edges in the graph. Each location on the board is converted into a linear vertex number. 

The `gen_legal_moves()` takes the position of the knight on the board and generates each of the eight possible moves, making sure those moves are still within the board:

```cpp
vector<pair<int, int>> genLegalMoves(int row, int col, int boardSize) {
    vector<pair<int, int>> newMoves;
    vector<pair<int, int>> moveOffsets = {
        {-1, -2}, {-1, 2}, {-2, -1}, {-2, 1},
        {1, -2},  {1, 2},  {2, -1},  {2, 1}};
    for (auto& off : moveOffsets) {
        int newRow = row + off.first;
        int newCol = col + off.second;
        if (0 <= newRow && newRow < boardSize && 0 <= newCol && newCol < boardSize) {
            newMoves.push_back({newRow, newCol});
        }
    }
    return newMoves;
}
```

<center><img src="https://raw.githubusercontent.com/phonchi/nsysu-math208/refs/heads/main/static_files/presentations/imgs/bigknight.png" width="45%" /></center>

The figure shows the complete graph of possible moves on an $8 \times 8$ board. There are exactly 336 edges in the graph. Notice that the vertices corresponding to the edges of the board have fewer connections (legal moves) than the vertices in the middle of the board.

Once again we can see how sparse the graph is. If the graph was fully connected there would be 4,096 edges. Since there are only 336 edges, the adjacency matrix would be only 8.2 percent full!

## 9.13. Implementing Knight’s Tour

The search algorithm we will use to solve the knight's tour problem is called <u>depth-first search (DFS)</u>. Whereas the breadth-first search algorithm builds a **search tree** one level at a time, a depth-first search creates a search tree by exploring one branch of the tree as deeply as possible.

We will look at two algorithms that implement DFS. The first algorithm we will look at specifically solves the knight's tour problem by explicitly **forbidding a node to be visited more than once.** 

The second implementation is more general, but allows nodes to be visited more than once as the tree is constructed. The second version is used in subsequent sections to develop additional graph algorithms.

The depth-first exploration of the graph is exactly what we need in order to find a path through 64 vertices (one for each square on the chessboard) and 63 edges.

We will see that when the depth-first search algorithm finds a dead end (a place in the graph where there are no more moves possible) it backs up the tree to the next deepest vertex that allows it to make a legal move.

The `knight_tour()` below takes four parameters: `n`, the current depth in the search tree; path, a `list` of vertices visited up to this point; `u`, the vertex in the graph we wish to explore; and `limit`, the number of nodes in the path. 

```cpp
bool knightTour(int n, vector<string>& path, string uKey, int limit, Graph& g) {
    g.vertices[uKey].color = "gray";
    path.push_back(uKey);
    if (n < limit) {
        bool done = false;
        for (auto& nb : g.vertices[uKey].neighbors) {
            if (done) break;
            if (g.vertices[nb.first].color == "white") {
                done = knightTour(n + 1, path, nb.first, limit, g);
            }
        }
        if (!done) {   // prepare to backtrack
            path.pop_back();
            g.vertices[uKey].color = "white";
        }
        return done;
    }
    return true;
}
```

The function is recursive. When the function is called, it first checks the base case condition. If we have a path that contains 64 vertices, we return from `knight_tour()` with a status of `True`, indicating that we have found a successful tour. If the path is not long enough, we continue to explore one level deeper by choosing a new vertex to explore and calling it recursively for that vertex.

DFS also uses colors to keep track of which vertices in the graph have been visited. Unvisited vertices are colored white, and visited vertices are colored gray. If all neighbors of a particular vertex have been explored and we have not yet reached our goal length of 64 vertices, we have reached a dead end and must <u>backtrack</u>. 

Backtracking happens when we return from `knight_tour()` with a status of `False`. In the breadth-first search we used a queue to keep track of which vertex to visit next. Since depth-first search is recursive, we are implicitly using a **stack** to help us with our backtracking.

When we return from a call with a status of `False`, we remain inside the `while` loop and look at the next vertex in neighbors.

Let's look at a simple example of `knight_tour()` in action. You can refer to the figures below to follow the steps of the search. For this example, we will assume that the call to the `get_neighbors()` method on line 5 orders the nodes in alphabetical order. We begin by calling `knight_tour(1, path, A, 6)`.

<center>
    <img src="https://raw.githubusercontent.com/phonchi/nsysu-math208/refs/heads/main/static_files/presentations/imgs/ktdfsa.png" width="27%"/>
</center>

<center>
    <img src="https://raw.githubusercontent.com/phonchi/nsysu-math208/refs/heads/main/static_files/presentations/imgs/ktdfsb.png" width="27%"/>
</center>

<center>
    <img src="https://raw.githubusercontent.com/phonchi/nsysu-math208/refs/heads/main/static_files/presentations/imgs/ktdfsc.png" width="27%"/>
</center>

<center>
    <img src="https://raw.githubusercontent.com/phonchi/nsysu-math208/refs/heads/main/static_files/presentations/imgs/ktdfse.png" width="27%"/>
</center>

<center>
    <img src="https://raw.githubusercontent.com/phonchi/nsysu-math208/refs/heads/main/static_files/presentations/imgs/ktdfsf.png" width="27%"/>
</center>

<center>
    <img src="https://raw.githubusercontent.com/phonchi/nsysu-math208/refs/heads/main/static_files/presentations/imgs/ktdfsg.png" width="27%"/>
</center>

`knight_tour()` starts with node A. The nodes adjacent to A are B and D. Since B is before D alphabetically, DFS selects B to expand next. Exploring B happens when `knight_tour()` is called recursively. B is adjacent to C and D, so it selects to explore C next.

However, as you can see node C is a dead end with no adjacent white nodes. At this point we change the color of node C back to white. The call to `knight_tour()` returns a value of `False`. The return from the recursive call effectively backtracks the search to vertex B. The next vertex on the list to explore is vertex D, so knight_tour makes a recursive call moving to node D.

From vertex D on, `knight_tour()` can continue to make recursive calls until we get to node C again. However, this time when we get to node C the test `n < limit` fails so we know that we have exhausted all the nodes in the graph. At this point we can return `True` to indicate that we have made a successful tour of the graph.

When we return the list, `path` has the values `[A, B, D, E, F, C]`, which is the order we need to traverse the graph to visit each node exactly once.

In [5]:
#include <iostream>
#include "dscpp/graph_algos.hpp"   // genLegalMoves + knightGraph + knightTour
using namespace std;

int main() {
    // brute force: 5x5 board - 8x8 without the heuristic takes far too long!
    int boardSize = 5;
    Graph ktGraph = knightGraph(boardSize);
    vector<string> path;
    bool finished = knightTour(1, path, "0", boardSize * boardSize, ktGraph);
    if (finished) printBoard(path, boardSize);   // squares show the move number
    else cout << "No path found." << endl;
    return 0;
}

   0   5  14  11  20

  15  10  19   6  13

   4   1  12  21  18

   9  16  23   2   7

  24   3   8  17  22



Instead of a matplotlib figure, we print the board as a grid where each square shows the move number on which the knight visits it — `0` is the start and 24 (or 63) the final move.

For an 8×8 board this plain depth-first search is hopeless — the next section explains why, and the *Warnsdorff heuristic* below fixes it.

Each vertex of the knight graph also carries the usual BFS/DFS bookkeeping state (color, previous), which the tour uses to avoid revisiting squares.

## 9.14. Knight's Tour Analysis

`knight_tour()` is very sensitive to the method you use to select the next vertex to visit. For example, on a $5 \times 5$ board you can produce a path in about 1.5 seconds on a reasonably fast computer. But what happens if you try an $8 \times 8$ board? In this case, depending on the speed of your computer, you may have to wait up to several minutes to get the results!

The reason for this is that the knight's tour problem as we have implemented it so far is an exponential algorithm of size $O(k^N)$, where $N$ is the number of squares on the chess board, and $k$ is a small constant.

We can use a tree to help us undertand this point. The root of the tree represents the starting point of the search. From there the algorithm generates and checks each of the possible moves the knight can make. 

As we have noted before, the number of moves possible depends on the position of the knight on the board. In the corners there are only two legal moves, on the squares adjacent to the corners there are three, and in the middle of the board there are eight.

<center>
    <img src="https://raw.githubusercontent.com/phonchi/nsysu-math208/refs/heads/main/static_files/presentations/imgs/moveCount.png" width="30%"/>
</center>

At the next level of the tree there are once again between two and eight possible next moves from the position we are currently exploring. The number of possible positions to examine corresponds to the number of nodes in the search tree.

<center>
    <img src="https://raw.githubusercontent.com/phonchi/nsysu-math208/refs/heads/main/static_files/presentations/imgs/8arrayTree.png" width="50%"/>
</center>

For a tree with nodes that may have up to eight children, the number of nodes is large. Because the branching factor of each node is variable, we could estimate the number of nodes using an **average branching factor**. The important thing to note is that this algorithm is exponential: $\frac{k^{N+1}-1}{k-1}$, where $k$ is the average branching factor for the board.

Let's look at how rapidly this grows! For a board that is $5 \times 5$ the tree will be 25 levels deep. The average branching factor is $k=3.8$ so the number of nodes in the search tree is $3.8^{25}-1$ or $3.12 \times 10^{14}$.

For a $6 \times 6$ board, $k = 4.4$, there are $1.5 \times 10^{23}$ nodes, and for a regular $8 \times 8$ chess board, $k = 5.25$, there are $1.2 \times 10^{46}$!

Of course, since there are multiple solutions to the problem we won't have to explore every single node, but the fractional part of the nodes we do have to explore is just a constant multiplier which does not change the exponential nature of the problem.

Luckily there is a way to speed up the $8 \times 8$ case so that it runs in under one second. The `order_by_avail()` will be used in place of the call to `u.get_neighbors()`. The critical line in the function is line 5. This line ensures that we select the vertex that has the fewest available moves to go next.

```cpp
vector<string> orderByAvail(Graph& g, string uKey) {
    vector<pair<int, string>> resList;
    for (auto& nb : g.vertices[uKey].neighbors) {
        if (g.vertices[nb.first].color == "white") {
            int c = 0;
            for (auto& w : g.vertices[nb.first].neighbors) {
                if (g.vertices[w.first].color == "white") {
                    c++;
                }
            }
            resList.push_back({c, nb.first});
        }
    }
    sort(resList.begin(), resList.end());   // fewest onward moves first
    vector<string> result;
    for (auto& p : resList) {
        result.push_back(p.second);
    }
    return result;
}
```

You might think this is really counterproductive; why not select the node that has the most available moves? 

You can try that approach easily by running the program yourself and inserting the line `res_list.reverse()` right after the sort.

The problem with using the vertex with the most available moves as your next vertex on the path is that it tends to have the knight visit the middle squares early on in the tour!

When this happens it is easy for the knight to get stranded on one side of the board where it cannot reach unvisited squares on the other side of the board. On the other hand, visiting the squares with the fewest available moves first pushes the knight to visit the squares around the edges of the board first.

This ensures that the knight will visit the hard-to-reach corners early and can use the middle squares to **hop across the board only when necessary.** Utilizing this kind of knowledge to speed up an algorithm is called a <u>heuristic</u>. This particular heuristic is called <u>Warnsdorff's algorithm</u>.

```cpp
bool knightTour(int n, vector<string>& path, string uKey, int limit, Graph& g) {
    g.vertices[uKey].color = "gray";
    path.push_back(uKey);
    if (n < limit) {
        bool done = false;
        for (string nbKey : orderByAvail(g, uKey)) {   // Warnsdorff order
            if (done) break;
            if (g.vertices[nbKey].color == "white") {
                done = knightTour(n + 1, path, nbKey, limit, g);
            }
        }
        if (!done) {
            path.pop_back();
            g.vertices[uKey].color = "white";
        }
        return done;
    }
    return true;
}
```

In [6]:
#include <iostream>
#include "dscpp/graph_algos.hpp"   // orderByAvail + knightTourWarnsdorff
using namespace std;

int main() {
    int boardSize = 8;   // with the heuristic, 8x8 finishes instantly
    Graph ktGraph = knightGraph(boardSize);
    vector<string> path;
    bool finished = knightTourWarnsdorff(1, path, "0", boardSize * boardSize, ktGraph);
    if (finished) printBoard(path, boardSize);
    else cout << "No path found." << endl;
    return 0;
}

   0   3  40  19  38   5  42  21

  33  18   1   4  41  20  37   6

   2  51  34  39  36  55  22  43

  17  32  49  62  53  44   7  56

  50  13  52  35  48  57  54  23

  31  16  61  58  63  26  45   8

  12  59  14  29  10  47  24  27

  15  30  11  60  25  28   9  46



With Warnsdorff's heuristic the 8×8 tour completes in milliseconds — a dramatic contrast with the exponential brute-force search.

## 9.15. General Depth-First Search

The knight's tour is a special case of a depth-first search where the goal is to create the deepest depth-first tree **without any branches**.

The more general depth-first search is actually easier. Its goal is to search as deeply as possible, connecting as many nodes in the graph as possible and branching where necessary.

It is even possible that a depth-first search will create more than one tree. When the depth-first search algorithm creates a group of trees we call this a **depth-first forest**. 

As with the breadth-first search, our depth-first search makes use of predecessor links to construct the tree. In addition, the depth-first search will make use of two additional instance variables in the `Vertex` class. The new instance variables are the discovery and closing times.

- The discovery time tracks the number of steps in the algorithm before a vertex is first encountered. 
- The closing time is the number of steps in the algorithm before a vertex is colored black.

As we will see after looking at the algorithm, the discovery and closing times of the nodes provide some interesting properties we can use in later algorithms.

```cpp
class DFSGraph : public Graph {
    public:
        int time = 0;
        map<string, int> discovery;
        map<string, int> closing;

        void dfs() {
            for (auto& p : vertices) {
                p.second.color = "white";
                p.second.previous = "";
            }
            for (auto& p : vertices) {
                if (p.second.color == "white") {
                    dfsVisit(p.first);
                }
            }
        }
    ...
```

```cpp
    ...
        void dfsVisit(string startKey) {
            vertices[startKey].color = "gray";
            time = time + 1;
            discovery[startKey] = time;
            for (auto& n : vertices[startKey].neighbors) {
                if (vertices[n.first].color == "white") {
                    vertices[n.first].previous = startKey;
                    dfsVisit(n.first);
                }
            }
            vertices[startKey].color = "black";
            time = time + 1;
            closing[startKey] = time;
        }
};
```

Note since that `dfs()` and its helper `dfs_visit()` use a variable to keep track of the time across calls to `dfs_visit()`, we chose to implement the code as methods of a class that inherits from the Graph class. This implementation extends the graph class by adding a `time` instance variable and the two methods.

You will notice that the `dfs()` method iterates over all of the vertices in the graph calling `dfs_visit()` on the nodes that are white.

The reason we iterate over all the nodes, rather than simply searching from a chosen starting node, is to make sure that all nodes in the graph are considered and that no vertices are left out of the depth-first forest.

It may look unusual to see the statement `for vertex in self`, but remember that in this case `self` is an instance of the `DFSGraph` class, and iterating over all the vertices in an instance of a graph is a natural thing to do.

The `dfs_visit()` method starts with a single vertex called `start` and explores all of the neighboring white vertices as deeply as possible. 

If you look carefully at the code for `dfs_visit()` and compare it to breadth-first search, what you should notice is that the algorithm is almost identical to `bfs()` except that on the last line of the inner for loop, `dfs_visit()` calls itself recursively to continue the search at a deeper level, whereas `bfs()` adds the node to a queue for later exploration!

It is interesting to note that where `bfs()` uses a queue, `dfs_visit()` uses a stack. You don't see a stack in the code, but it is implicit in the recursive call.

The following sequence of figures illustrates the depth-first search algorithm in action for a small graph. In these figures, the dotted lines indicate edges that are checked, but the node at the other end of the edge has already been added to the depth-first tree.

<center>
    <img src="https://raw.githubusercontent.com/phonchi/nsysu-math208/refs/heads/main/static_files/presentations/imgs/gendfsa.png" width="27%"/>
</center>

<center>
    <img src="https://raw.githubusercontent.com/phonchi/nsysu-math208/refs/heads/main/static_files/presentations/imgs/gendfsb.png" width="27%"/>
</center>

<center>
    <img src="https://raw.githubusercontent.com/phonchi/nsysu-math208/refs/heads/main/static_files/presentations/imgs/gendfsc.png" width="27%"/>
</center>

<center>
    <img src="https://raw.githubusercontent.com/phonchi/nsysu-math208/refs/heads/main/static_files/presentations/imgs/gendfsd.png" width="27%"/>
</center>

<center>
    <img src="https://raw.githubusercontent.com/phonchi/nsysu-math208/refs/heads/main/static_files/presentations/imgs/gendfse.png" width="27%"/>
</center>

<center>
    <img src="https://raw.githubusercontent.com/phonchi/nsysu-math208/refs/heads/main/static_files/presentations/imgs/gendfsf.png" width="27%"/>
</center>

<center>
    <img src="https://raw.githubusercontent.com/phonchi/nsysu-math208/refs/heads/main/static_files/presentations/imgs/gendfsg.png" width="27%"/>
</center>

<center>
    <img src="https://raw.githubusercontent.com/phonchi/nsysu-math208/refs/heads/main/static_files/presentations/imgs/gendfsh.png" width="27%"/>
</center>

<center>
    <img src="https://raw.githubusercontent.com/phonchi/nsysu-math208/refs/heads/main/static_files/presentations/imgs/gendfsi.png" width="27%"/>
</center>

<center>
    <img src="https://raw.githubusercontent.com/phonchi/nsysu-math208/refs/heads/main/static_files/presentations/imgs/gendfsj.png" width="27%"/>
</center>

<center>
    <img src="https://raw.githubusercontent.com/phonchi/nsysu-math208/refs/heads/main/static_files/presentations/imgs/gendfsk.png" width="27%"/>
</center>

<center>
    <img src="https://raw.githubusercontent.com/phonchi/nsysu-math208/refs/heads/main/static_files/presentations/imgs/gendfsl.png" width="27%"/>
</center>

The search begins at vertex A of the graph. Since all of the vertices are white at the beginning of the search the algorithm visits vertex A. The first step in visiting a vertex is to set the color to gray, which indicates that the vertex is being explored, and the discovery time is set to 1. Since vertex A has two adjacent vertices (B, D) each of those need to be visited as well. We'll make the arbitrary decision that we will visit the adjacent vertices in alphabetical order.

Vertex B is visited next, so its color is set to gray and its discovery time is set to 2. Vertex B is also adjacent to two other nodes (C, D) so we will follow the alphabetical order and visit node C next.

Visiting vertex C brings us to the end of one branch of the tree. After coloring the node gray and setting its discovery time to 3, the algorithm also determines that there are no adjacent vertices to C. This means that we are done exploring node C and so we can color the vertex black and set the closing time to 4.

Since vertex C is the end of one branch, we now return to vertex B and continue exploring the nodes adjacent to B. The only additional vertex to explore from B is D, so we can now visit D and continue our search from vertex D. Vertex D quickly leads us to vertex E. Vertex E has two adjacent vertices, B and F. Normally we would explore these adjacent vertices alphabetically, but since B is already colored gray the algorithm recognizes that it should not visit B since doing so would put the algorithm in a loop! So exploration continues with the next vertex in the list, namely F.

Vertex F has only one adjacent vertex, C, but since C is colored black there is nothing else to explore, and the algorithm has reached the end of another branch. From here on, you will see the algorithm works its way back to the first node, setting closing times and coloring vertices black.

The discovery and closing times for each node display a property called the <u>parenthesis property</u>. This property means that all the children of a particular node in the depth-first tree have a later discovery time and an earlier closing time than their parent.

In [7]:
#include <iostream>
#include "dscpp/graph_algos.hpp"   // DFSGraph (md listing above)
using namespace std;

int main() {
    DFSGraph g;
    g.addEdge("A", "B");  g.addEdge("B", "C");
    g.addEdge("A", "D");  g.addEdge("B", "D");
    g.addEdge("D", "E");  g.addEdge("E", "B");
    g.addEdge("E", "F");  g.addEdge("F", "C");
    g.dfs();
    printf("%4s|%9s|%8s|%9s\n", "Key", "Discover", "Closing", "Previous");
    for (auto& p : g.vertices)
        printf("%4s|%9d|%8d|%9s\n", p.first.c_str(),
               g.discovery[p.first], g.closing[p.first], p.second.previous.c_str());
    return 0;
}

 Key| Discover| Closing| Previous

   A|        1|      12|         

   B|        2|      11|        A

   C|        3|       4|        B

   D|        5|      10|        B

   E|        6|       9|        D

   F|        7|       8|        E



Every vertex ends up black, with its discovery/closing times bracketing those of its descendants — the *parenthesis property*.

Iterating over the graph lists each directed edge exactly once, confirming the adjacency structure.

In [ ]:
load_anim("dfs")

## 9.16. Depth-First Search Analysis

The general running time for depth-first search is as follows. The loops in `dfs()` run in $O|V|$, not counting what happens in `dfs_visit()`, since it is executed once for each vertex in the graph.

 In `dfs_visit()` the loop is executed once for each edge in the adjacency list of the current vertex. Since it is only called recursively if the vertex is white, the loop will execute a maximum of once for every edge in the graph, or $O(E)$. Therefore, the total time for depth-first search is $O(|V| + |E|)$.

In [ ]:
display_quiz(quiz_path+"bfs_dfs.json", max_width=800)

<IPython.core.display.Javascript object>

## 9.17. Topological Sorting (Optional)

To demonstrate that computer scientists can turn just about anything into a graph problem, let’s consider the difficult problem of stirring up a batch of pancakes.

The recipe is really quite simple: 1 egg, 1 cup of pancake mix, 1 tablespoon oil, and $3 \over 4$ cup of milk. To make pancakes you must heat the griddle, mix all the ingredients together, and spoon the mix onto a hot griddle.

When the pancakes start to bubble you turn them over and let them cook until they are golden brown on the bottom. Before you eat your pancakes you are going to want to heat up some syrup.

<center><img src="https://raw.githubusercontent.com/phonchi/nsysu-math208/refs/heads/main/static_files/presentations/imgs/pancakes.png" width="33%" /></center>

The difficult thing about making pancakes is knowing what to do first. As you can see, you might start by heating the griddle or by adding any of the ingredients to the pancake mix. To help us decide the precise order in which we should do each of the steps required to make our pancakes, we turn to a graph algorithm called the <u>topological sort</u>.

A topological sort takes a DAG and produces a linear ordering of all its vertices such that if the graph contains an edge $(v,w)$ then the vertex $v$ comes before the vertex $w$ in the ordering.

Directed acyclic graphs are used in many applications to indicate the **precedence of events**. Making pancakes is just one example; other examples include software project schedules, precedence charts for optimizing database queries, and multiplying matrices.

The topological sort is a simple but useful adaptation of a depth-first search. The algorithm for the topological sort is as follows:

1. Call `dfs(g)` for some graph `g`. The main reason we want to call depth-first search is to compute the closing times for each of the vertices.

2. Store the vertices in a list in decreasing order of the closing time.

3. Return the ordered list as the result of the topological sort.

Following shows the depth-first forest constructed by dfs on the pancake-making graph:

<center><img src="https://raw.githubusercontent.com/phonchi/nsysu-math208/refs/heads/main/static_files/presentations/imgs/pancakesDFS.png" width="33%" /></center>

Figure below shows the results of applying the topological sort algorithm to our graph. Now all the ambiguity has been removed and we know exactly the order in which to perform the pancake-making steps.

<center><img src="https://raw.githubusercontent.com/phonchi/nsysu-math208/refs/heads/main/static_files/presentations/imgs/pancakesTS.png" width="83%" /></center>

In [ ]:
load_anim("topsort")

## 9.18. Strongly Connected Components (Optional)

For the remainder of this chapter we will turn our attention to some extremely large graphs. The graphs we will use to study some additional algorithms are the graphs produced by the connections between hosts on the internet and the links between web pages.

Search engines like Google and Bing exploit the fact that the pages on the web form a very large directed graph. To transform the World Wide Web into a graph, we will treat a page as a vertex, and the hyperlinks on the page as edges connecting one vertex to another

You might make some interesting observations on these kind of graphs.You might conclude from this that there is some underlying structure to the Web that clusters together websites that are similar on some level.

One graph algorithm that can help find clusters of highly interconnected vertices in a graph is called the <u>strongly connected components</u> algorithm, or SCC. We formally define a strongly connected component $C$ as the largest subset of vertices $C \subset V$ such that for every pair of vertices $v, w \in C$ we have a path from $v$ to $w$ and a path from $w$ to $v$.

Figure below shows a simple graph with three strongly connected components that are identified by the different shaded areas.

<center><img src="https://raw.githubusercontent.com/phonchi/nsysu-math208/refs/heads/main/static_files/presentations/imgs/scc1.png" width="35%" /></center>

Once the strongly connected components have been identified, we can show a simplified view of the graph by combining all the vertices in one strongly connected component into a single larger vertex.

<center><img src="https://raw.githubusercontent.com/phonchi/nsysu-math208/refs/heads/main/static_files/presentations/imgs/scc2.png" width="35%" /></center>

We will see that we can create a very powerful and efficient algorithm by making use of a depth-first search. Before we tackle the main SCC algorithm we must look at one other definition. The transposition of a graph is defined as the graph $G^T$ where all the edges in the graph have been reversed. 

That is, if there is a directed edge from node A to node B in the original graph, then $G^T$ will contain an edge from node B to node A

We can now describe the algorithm to compute the strongly connected components for a graph:

1. Call `dfs()` for the graph $G$ to compute the closing times for each vertex.

2. Compute $G^T$.

3. Call `dfs()` for the graph $G^T$ but in the main loop of DFS explore each vertex in decreasing order of closing time.

4. Each tree in the forest computed in step 3 is a strongly connected component. Output the vertex IDs for each vertex in each tree in the forest to identify the component.

Let's trace the operation of the steps described above on the example graph above.

Below shows the starting and closing times computed for the original graph by the DFS algorithm.

<center><img src="https://raw.githubusercontent.com/phonchi/nsysu-math208/refs/heads/main/static_files/presentations/imgs/scc1a.png" width="45%" /></center>

Below shows the starting and closing times computed for the transposed graph by the DFS algorithm.

<center><img src="https://raw.githubusercontent.com/phonchi/nsysu-math208/refs/heads/main/static_files/presentations/imgs/scc1b.png" width="45%" /></center>

Finally, figure below shows the forest of three trees produced in step 3 of the strongly connected components algorithm.

<center><img src="https://raw.githubusercontent.com/phonchi/nsysu-math208/refs/heads/main/static_files/presentations/imgs/sccforest.png" width="45%" /></center>

## 9.19. Shortest Path Problems

When you surf the web, send an email, or log in to a laboratory computer from another location on campus, a lot of work is going on behind the scenes to get the information on your computer transferred to another computer. The in-depth study of how information flows from one computer to another over the internet is the primary topic for a class in computer networking.

<center><img src="https://raw.githubusercontent.com/phonchi/nsysu-math208/refs/heads/main/static_files/presentations/imgs/Internet.png" width="45%" /></center>

Figure above shows you a high-level overview of how communication on the internet works. When you use your browser to request a web page from a server, the request must travel over your local area network and out onto the internet through a router. 

Each router on the internet is connected to one or more other routers. If you run the `traceroute` command at different times of the day, you are likely to see that your information flows through different routers at different times. This is because there is a cost associated with each connection between a pair of routers that depends on the volume of traffic, the time of day, and many other factors.

By this time it will not surprise you to learn that we can represent the network of routers as a graph with weighted edges.

Figure below shows a small example of a weighted graph that represents the interconnection of routers in the internet. The problem that we want to solve is to find the shortest path, one with the smallest total weight along which to route any given message.

This problem should sound familiar because it is similar to the problem we solved using a breadth-first search, except that here we are concerned with the total weight of the path rather than the number of hops in the path. It should be noted that if all the weights are equal, the problem is the same!

<center><img src="https://raw.githubusercontent.com/phonchi/nsysu-math208/refs/heads/main/static_files/presentations/imgs/routeGraph.png" width="35%" /></center>

In [ ]:
display_quiz(quiz_path+"bfs.json", max_width=800)

<IPython.core.display.Javascript object>

## 9.20. Dijkstra's Algorithm

<u>Dijkstra's algorithm</u> is an iterative algorithm that provides us with the shortest path from one particular starting node to all other nodes in the graph. Again this is similar to the results of a breadth-first search.

To keep track of the total cost from the start node to each destination, we will make use of the `distance` instance variable in the `Vertex` class. The `distance` instance variable will contain the current total weight of the smallest weight path from the start to the vertex in question. 

The algorithm iterates once for every vertex in the graph; however, the order that it iterates over the vertices is controlled by a <u>priority queue</u>. The value that is used to determine the order of the objects in the priority queue is **distance**.

A priority queue acts like a queue in that you dequeue an item by removing it from the front. However, in a priority queue the logical order of items inside a queue is determined by their priority. The highest priority items are at the front of the queue and the lowest priority items are at the back. 

Thus when you enqueue an item on a priority queue, the new item may move all the way to the front.

When a vertex is first created, distance is set to a very large number. Theoretically you would set distance to infinity, but in practice we just set it to a number that is larger than any real distance we would have in the problem we are trying to solve.

In `C++` we use the standard `priority_queue` as our min-heap: it stores `(distance, key)` pairs and, with `greater<>`, always pops the smallest distance first.

```cpp
void dijkstra(Graph& g, string startKey) {
    // min-heap of (distance, vertex key) pairs
    priority_queue<pair<int, string>,
                   vector<pair<int, string>>,
                   greater<pair<int, string>>> pq;
    g.vertices[startKey].distance = 0;
    pq.push({0, startKey});

    while (!pq.empty()) {
        auto [distance, currentKey] = pq.top();
        pq.pop();
        if (distance > g.vertices[currentKey].distance) {
            continue;   // stale entry - the C++ version of change_priority
        }
        for (auto& n : g.vertices[currentKey].neighbors) {
            int newDistance = g.vertices[currentKey].distance + n.second;
            if (newDistance < g.vertices[n.first].distance) {
                g.vertices[n.first].distance = newDistance;
                g.vertices[n.first].previous = currentKey;
                pq.push({newDistance, n.first});
            }
        }
    }
}
```

Instead of a `change_priority` operation, the standard C++ idiom pushes a **new** entry with the better distance and skips any *stale* entries when they surface — same asymptotic complexity, much simpler code. A `visited` set variant terminates correctly as long as no edge is negative.

The `PriorityQueue` class stores tuples of `(priority, key)` pairs. This is an important point, because Dijkstra's algorithm requires the key in the priority queue to match the key of the vertex in the graph. 

The priority is used for deciding the position of the key in the priority queue. In this implementation we use the distance to the vertex as the priority because as we will see when we are exploring the next vertex, we always want to explore the vertex that has the smallest distance. 

Note that there is a `change_priority()` method. This method is used when the distance to a vertex that is already in the queue is reduced, and thus the vertex is moved toward the front of the queue!

Let's walk through it. We begin with the vertex $u$. The three vertices adjacent to it are $v,w$ and $x$. Since the initial distances to them are all initialized to `sys.maxsize`, the new costs to get to them through the start node are all their direct costs. So we update the costs to each of these three nodes. We also set the predecessor for each node to $u$
 and we add each node to the priority queue. We use the distance as the key for the priority queue. 

<center><img src="https://raw.githubusercontent.com/phonchi/nsysu-math208/refs/heads/main/static_files/presentations/imgs/dijkstraa.png" width="33%" /></center>

<center><img src="https://raw.githubusercontent.com/phonchi/nsysu-math208/refs/heads/main/static_files/presentations/imgs/dijkstrab.png" width="33%" /></center>

<center><img src="https://raw.githubusercontent.com/phonchi/nsysu-math208/refs/heads/main/static_files/presentations/imgs/dijkstrac.png" width="33%" /></center>

<center><img src="https://raw.githubusercontent.com/phonchi/nsysu-math208/refs/heads/main/static_files/presentations/imgs/dijkstrad.png" width="35%"3/></center>

<center><img src="https://raw.githubusercontent.com/phonchi/nsysu-math208/refs/heads/main/static_files/presentations/imgs/dijkstrae.png" width="33%" /></center>

<center><img src="https://raw.githubusercontent.com/phonchi/nsysu-math208/refs/heads/main/static_files/presentations/imgs/dijkstraf.png" width="33%" /></center>

In the next iteration of the `while` loop we examine the vertices that are adjacent to $x$. The vertex $x$ is next because it has the lowest overall cost and therefore bubbled its way to the beginning of the priority queue. At $x$ we look at its neighbors $u,v,w$ and $y$. 

For each neighboring vertex we check to see if the distance to that vertex through $x$ is smaller than the previously known distance. Obviously this is the case for $y$ since its distance was `sys.maxsize`. It is not the case for $u$ or $v$ since their distances are 0 and 2 respectively. However, we now learn that the distance to $w$ is smaller if we go through $x$ 
 than from $u$ directly to $w$. Since that is the case we update $w$ with a new distance and change the predecessor for $w$
 from $u$ to $x$.

The next step is to look at the vertices neighboring $v$. This step results in no changes to the graph, so we move on to node $y$. At node $y$ we discover that it is cheaper to get to both $w$ and $z$, so we adjust the distances and predecessor links accordingly. Finally we check nodes $w$ and $z$. However, no additional changes are found and so the priority queue is empty and Dijkstra's algorithm exits.

In [8]:
#include <iostream>
#include "dscpp/graph_algos.hpp"   // dijkstra (md listing above)
using namespace std;

int main() {
    Graph g;
    g.addEdge("u","v",2); g.addEdge("v","u",2); g.addEdge("v","w",3); g.addEdge("w","v",3);
    g.addEdge("w","z",5); g.addEdge("z","w",5); g.addEdge("u","x",1); g.addEdge("x","u",1);
    g.addEdge("u","w",5); g.addEdge("w","u",5); g.addEdge("x","v",2); g.addEdge("v","x",2);
    g.addEdge("x","w",3); g.addEdge("w","x",3); g.addEdge("x","y",1); g.addEdge("y","x",1);
    g.addEdge("y","w",1); g.addEdge("w","y",1); g.addEdge("y","z",1); g.addEdge("z","y",1);
    dijkstra(g, "u");
    for (auto& p : g.vertices)   // key: distance (previous)
        cout << p.first << ": " << p.second.distance
             << " (" << p.second.previous << ")" << endl;
    return 0;
}

u: 0 ()

v: 2 (u)

w: 3 (y)

x: 1 (u)

y: 2 (x)

z: 3 (y)



Every vertex now carries the length of the **shortest** path from `u`, and following `previous` pointers backwards reconstructs that path.

In [9]:
#include <iostream>
#include "dscpp/graph_algos.hpp"   // dijkstra + findPath
using namespace std;

int main() {
    Graph g;
    g.addEdge("u","v",2); g.addEdge("v","u",2); g.addEdge("v","w",3); g.addEdge("w","v",3);
    g.addEdge("w","z",5); g.addEdge("z","w",5); g.addEdge("u","x",1); g.addEdge("x","u",1);
    g.addEdge("u","w",5); g.addEdge("w","u",5); g.addEdge("x","v",2); g.addEdge("v","x",2);
    g.addEdge("x","w",3); g.addEdge("w","x",3); g.addEdge("x","y",1); g.addEdge("y","x",1);
    g.addEdge("y","w",1); g.addEdge("w","y",1); g.addEdge("y","z",1); g.addEdge("z","y",1);
    dijkstra(g, "u");
    for (string v : findPath(g, "u", "z")) cout << v << " ";
    cout << endl;
    return 0;
}

u x y z 



It is important to note that Dijkstra's algorithm works only when the weights are all **positive**. You should convince yourself that if you introduced a negative weight on one of the edges of the graph, the algorithm would never exit.

We will note that to route messages through the internet, other algorithms are used for finding the shortest path. One of the problems with using Dijkstra's algorithm on the internet is that you must have a complete representation of the graph in order for the algorithm to run. The implication of this is that every router has a complete map of all the routers in the internet!

In practice this is not the case and other variations of the algorithm allow each router to discover the graph as they go.

In [ ]:
load_anim("dijkstra")

https://visualgo.net/en/sssp

## 9.21. Analysis of Dijkstra's Algorithm

Let's look at the running time of Dijkstra's algorithm. We first note that building the priority queue takes $O(|V|)$ time since we initially add every vertex in the graph to the priority queue (or in our case we perform initialization). Once the queue is constructed, the `while` loop is executed once for every vertex since vertices are all added at the beginning and only removed after that.

 Within that loop each call to `delete()` takes $O(\log{|V|})$ time. Taken together, that part of the loop and the calls to `delete()` take $O(|V| \times \log{|V|})$. The `for` loop is executed once for each edge in the graph, and within the for loop the call to `change_priority()` takes $O(|E| \times \log{|V|})$ time. So the combined running time is $O((|V|+|E|) \times \log{|V|})$.

In [ ]:
display_quiz(quiz_path+"dijk.json", max_width=800)

<IPython.core.display.Javascript object>

#### Exercise 1: Given a weighted directed graph with negative edges as follows, what is the resulting shortest path from vertex A to D after performing Dijkstra's algorithm if we do check the visited node?

<center><img src="https://raw.githubusercontent.com/phonchi/nsysu-math208/refs/heads/main/static_files/presentations/imgs/Dij.png" width="25%" /></center>


(A) `A->B->D`

(B) `A->C->D`

(C) `A->E->G->D`

(D) `A->C->E->G->D`

## 9.22. Prim's Spanning Tree Algorithm

For our last graph algorithm let's consider a problem that online game designers. The problem is that they want to efficiently transfer a piece of information to anyone and everyone who may be listening. This is important in gaming so that all the players know the very latest position of every other player!

<center><img src="https://raw.githubusercontent.com/phonchi/nsysu-math208/refs/heads/main/static_files/presentations/imgs/bcast1.png" width="45%" /></center>

There are some brute force solutions to this problem. To begin, the broadcast host has some information that the listeners all need to receive. The simplest solution is for the broadcasting host to keep a list of all of the listeners and send individual messages to each.

In the above figure we show a small network with a broadcaster and some listeners. Using this first approach, four copies of every message would be sent. Assuming that the **least cost path** is used, let's see how many times each router would handle the same message.

All messages from the broadcaster go through router A, so A sees all four copies of every message. Router C sees only one copy of each message for its listener. However, routers B and D would see three copies of every message since routers B and D are on the **cheapest path** for listeners 1, 2, and 4! When you consider that the broadcast host must send hundreds of messages each second for a radio broadcast, that is a lot of extra traffic.

Another brute force solution is for the broadcast host to send a single copy of the broadcast message and let the routers sort things out. In this case, the easiest solution is a strategy called <u>uncontrolled flooding</u>:

Each message starts with a time to live (`TTL`) value set to some number greater than or equal to the number of edges between the broadcast host and its most distant listener. Each router gets a copy of the message and passes the message on to all of its neighboring routers. When the message is passed on the TTL is decreased. Because each router continues to send copies of the message to all its neighbors until the TTL value reaches 0, it is easy to convince yourself that uncontrolled flooding generates many more unnecessary messages than our first strategy.

The solution to this problem lies in the construction of a minimum weight <u>spanning tree</u>. Formally we define the minimum spanning tree $T$ for a graph $G = (V,E)$ as follows. $T$ is an acyclic subset of $E$ that connects all the vertices in $V$. The sum of the weights of the edges in $T$ is minimized.

Below shows a simplified version of the broadcast graph and highlights the edges that form a minimum spanning tree for the graph. 

<center><img src="https://raw.githubusercontent.com/phonchi/nsysu-math208/refs/heads/main/static_files/presentations/imgs/mst1.png" width="30%" /></center>

Now to solve our broadcast problem, the broadcast host simply sends a single copy of the broadcast message into the network. Each router forwards the message to any neighbor that is part of the spanning tree, excluding the neighbor that just sent it the message. 

In this example A forwards the message to B. B forwards the message to D and C. D forwards the message to E, which forwards it to F, which forwards it to G. No router sees more than one copy of any message, and all the listeners that are interested see a copy of the message.

The algorithm we will use to solve this problem is called <u>Prim's algorithm</u>. Prim's algorithm belongs to a family of algorithms called the <u>greedy algorithms</u> that applies to **undirected graph** because at each step it will choose the cheapest next step. In this case, the cheapest next step is to follow the edge with the lowest weight.

```
While T is not yet a spanning tree
  Find an edge that is safe to add to the tree
  Add the new edge to T
```

The trick is in the step that directs us to "find an edge that is safe." We define a safe edge as any edge that connects a vertex that is in the spanning tree to a vertex that is not in the spanning tree. This ensures that the tree will always remain a tree and therefore have no cycles.

```cpp
void prim(Graph& g, string startKey) {
    priority_queue<pair<int, string>,
                   vector<pair<int, string>>,
                   greater<pair<int, string>>> pq;
    set<string> inTree;

    for (auto& p : g.vertices) {
        p.second.distance = INT_MAX;
        p.second.previous = "";
    }
    g.vertices[startKey].distance = 0;
    pq.push({0, startKey});
    ...
```

```cpp
    ...
    while (!pq.empty()) {
        auto [distance, currentKey] = pq.top();
        pq.pop();
        if (inTree.count(currentKey)) continue;   // stale entry
        inTree.insert(currentKey);
        for (auto& n : g.vertices[currentKey].neighbors) {
            int newDistance = n.second;   // edge weight only, not path length!
            if (!inTree.count(n.first) && newDistance < g.vertices[n.first].distance) {
                g.vertices[n.first].distance = newDistance;
                g.vertices[n.first].previous = currentKey;
                pq.push({newDistance, n.first});
            }
        }
    }
}
```

*(complete listing: `dscpp/graph_algos.hpp`)*

In [ ]:
load_anim("prim")

The following sequence of figures shows the algorithm in operation on our sample tree. We begin with the starting vertex as A. The distances to all the other vertices are initialized to infinity. 

<center><img src="https://raw.githubusercontent.com/phonchi/nsysu-math208/refs/heads/main/static_files/presentations/imgs/prima.png" width="25%" /></center>

<center><img src="https://raw.githubusercontent.com/phonchi/nsysu-math208/refs/heads/main/static_files/presentations/imgs/primb.png" width="25%" /></center>

<center><img src="https://raw.githubusercontent.com/phonchi/nsysu-math208/refs/heads/main/static_files/presentations/imgs/primc.png" width="25%" /></center>

<center><img src="https://raw.githubusercontent.com/phonchi/nsysu-math208/refs/heads/main/static_files/presentations/imgs/primd.png" width="25%" /></center>

<center><img src="https://raw.githubusercontent.com/phonchi/nsysu-math208/refs/heads/main/static_files/presentations/imgs/prime.png" width="25%" /></center>

<center><img src="https://raw.githubusercontent.com/phonchi/nsysu-math208/refs/heads/main/static_files/presentations/imgs/primf.png" width="25%" /></center>

<center><img src="https://raw.githubusercontent.com/phonchi/nsysu-math208/refs/heads/main/static_files/presentations/imgs/primg.png" width="25%" /></center>

We begin with the starting vertex as A. The distances to all the other vertices are initialized to infinity. Looking at the neighbors of A we can update distances to two of the additional vertices, B and C, because the distances to B and C through A are less than infinite.

This moves B and C to the front of the priority queue. Update the predecessor links for B and C by setting them to point to A. It is important to note that we have not formally added B or C to the spanning tree yet. **A node is not considered to be part of the spanning tree until it is removed from the priority queue**.

Since B has the smallest distance we look at B next. Examining B's neighbors we see that D and E can be updated. Both D and E get new distance values and their predecessor links are updated. Moving on to the next node in the priority queue we find C. The only node that C is adjacent to that is still in the priority queue is F; thus we can update the distance to F and adjust F’s position in the priority queue.

Now we examine the vertices adjacent to node D. We find that we can update E and reduce the distance to E from 6 to 4. When we do this we change the predecessor link on E to point back to D, thus preparing it to be grafted into the spanning tree but in a different location. The rest of the algorithm proceeds as you would expect, adding each new node to the tree.

In [10]:
#include <iostream>
#include "dscpp/graph_algos.hpp"   // prim (md listing above)
using namespace std;

int main() {
    Graph g;
    g.addEdge("A","B",2); g.addEdge("B","A",2); g.addEdge("A","C",3); g.addEdge("C","A",3);
    g.addEdge("B","D",1); g.addEdge("D","B",1); g.addEdge("B","C",1); g.addEdge("C","B",1);
    g.addEdge("B","E",4); g.addEdge("E","B",4); g.addEdge("D","E",1); g.addEdge("E","D",1);
    g.addEdge("C","F",5); g.addEdge("F","C",5); g.addEdge("E","F",1); g.addEdge("F","E",1);
    g.addEdge("F","G",1); g.addEdge("G","F",1);

    prim(g, "A");
    cout << "MST edges: ";
    for (auto& p : g.vertices)
        if (p.second.previous != "")
            cout << "(" << p.second.previous << "," << p.first << ") ";
    cout << endl;
    return 0;
}

MST edges: (A,B) (B,C) (B,D) (D,E) (E,F) (F,G) 



The printed edges form the minimum spanning tree rooted at `A` — the same tree the Python version produced, discovered by always growing along the cheapest frontier edge.

In [ ]:
display_quiz(quiz_path+"prim.json", max_width=800)

<IPython.core.display.Javascript object>

https://visualgo.net/en/mst

#### Exercise 2: Considering the following graph, when Prim's algorithm is used and starts from node F, what is the weight of the last edge to be added into the minimum spanning tree?

<center><img src="https://raw.githubusercontent.com/phonchi/nsysu-math208/refs/heads/main/static_files/presentations/imgs/primq.png" width="25%" /></center>

In [ ]:
from jupytercards import display_flashcards
fpath = "flashcards/"
display_flashcards(fpath + 'ch8.json')

<IPython.core.display.Javascript object>

## References

1. Textbook: *Problem Solving with Algorithms and Data Structures using C++* (cppds), Chapter 9 (Graphs and Graph Algorithms) — https://runestone.academy/ns/books/published/cppds/index.html

## Key terms

Here's a brief definition for each of the key terms:

- **Graph**: A data structure composed of a set of vertices (nodes) and edges (links) that connect pairs of these vertices. It is used to model relationships in complex systems.

- **Vertex**: Also known as a node, it represents an entity within a graph and serves as the fundamental unit of the structure.

- **Edge**: A connection between two vertices. In a directed graph, edges indicate a specific direction of relationship.

- **Weight**: A numerical value assigned to an edge representing cost, distance, or another measure impacting traversal.

- **Path**: A sequence of vertices connected by edges, defined by starting at one vertex and traversing through edges to reach another.

- **Cycle**: A path that begins and ends at the same vertex, containing at least one edge and no repeated edges or internal vertices.

- **Acyclic Graph**: A graph with no cycles, meaning it is impossible to return to a starting vertex by following edges.

- **Directed Graph**: A graph where edges have a specific direction, indicating a one-way relationship from one vertex to another.

- **DAG (Directed Acyclic Graph)**: A directed graph containing no cycles, often used for task scheduling and sequences where tasks do not repeat.

- **Tree**: A type of acyclic graph with a designated root vertex where all other vertices are connected by exactly one path to that root.

- **Adjacency Matrix**: A square matrix used to represent a graph where elements indicate whether pairs of vertices are adjacent.

- **Adjacency List**: A collection of lists used to represent a graph, where each list describes the neighbors of a specific vertex.

- **Breadth-First Search (BFS)**: A search algorithm that explores all neighbor nodes at the current depth prior to moving on to nodes at the next depth level.

- **Knight's Tour**: A sequence of moves on a chessboard where a knight visits every square exactly once.

- **Depth-First Search (DFS)**: A traversal algorithm that explores as far as possible along each branch before backtracking.

- **Search Tree**: A tree data structure used for efficiently locating specific keys within a set through insertion, deletion, and search operations.

- **Heuristic**: A technique designed to solve problems quickly or find approximate solutions when classic methods are too slow or infeasible.

- **Warnsdorff's Algorithm**: A heuristic for the knight's tour where the knight always moves to the square with the fewest onward moves.

- **Depth-First Forest**: A collection of trees formed during a depth-first search, where each tree corresponds to a set of vertices explored contiguously.

- **Parenthesis Property**: A DFS property stating that discovery and finish time intervals for any two vertices are either entirely disjoint or nested.

- **Topological Sorting**: A linear ordering of vertices in a directed graph where for every edge uv, vertex u comes before v.

- **Strongly Connected Components**: Maximal subgraphs where every pair of vertices has a directed path to and from each other.

- **Shortest Path Problems**: The challenge of finding a path between two vertices such that the sum of the weights of the edges is minimized.

- **Dijkstra's Algorithm**: An algorithm for finding the shortest paths between nodes in a graph with non-negative edge weights.

- **Priority Queue**: A data structure where each element has a priority, and elements are served based on that priority rather than just arrival order.

- **Uncontrolled Flooding**: A networking technique where a packet is sent to every outgoing link except the one it arrived on.

- **Minimum Spanning Tree (MST)**: A subset of edges connecting all vertices without cycles and with the minimum total edge weight.

- **Prim's Algorithm**: An algorithm that finds a minimum spanning tree by growing a tree one edge at a time from a starting vertex.

- **Greedy Algorithms**: A class of algorithms that build a solution by always choosing the next piece that offers the most immediate benefit.
